# Criba de motores TTS — métricas y página de escucha

Se ejecuta **después** de los cuadernos 01–05; mide lo que haya en `outputs/criba/` aunque falte algún
motor. Reutiliza el método de tu `compare_xtts_vs_chatterbox`:

- **WER** con faster-whisper large-v3 en español: palabras que el audio no dice como debe. Detecta
  alucinaciones, saltos y repeticiones.
- **Similitud** con ECAPA-TDNN entre cada audio y la media de las 3 referencias de su voz, en inglés.
- **UTMOSv2**: calidad percibida estimada, de 1 a 5.

Después monta **`escucha.html`**, la página de escucha a ciegas: las versiones de cada frase aparecen
con letras al azar, y la clave para descifrarlas se guarda aparte, en `clave.json`.

**GPU:** T4. **Tiempo estimado:** unos 40 min. Mismo uso que los demás: celda 1, reiniciar sesión,
y el resto en orden. Lo ya medido no se vuelve a medir.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q faster-whisper==1.2.1 jiwer speechbrain soundfile librosa pandas
!pip install -q git+https://github.com/sarulab-speech/UTMOSv2.git

In [ ]:
# 2. Preparar
import os, sys, subprocess, json
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun
comun.montar_drive()
ITEMS, _ = comun.cargar_items()
POR_ID = {it["id"]: it for it in ITEMS}
METRICAS = comun.SALIDA / "metricas"
METRICAS.mkdir(parents=True, exist_ok=True)
VARIANTES = sorted(p.name for p in comun.SALIDA.iterdir() if (p / "registro.jsonl").exists())
AUDIOS = [(v, i) for v in VARIANTES for i in POR_ID if (comun.SALIDA / v / f"{i}.wav").exists()]
print("Variantes:", VARIANTES)
print("Audios a medir:", len(AUDIOS))

In [ ]:
# 3. WER con faster-whisper large-v3
import re, jiwer
import numpy as np, soundfile as sf, librosa
from faster_whisper import WhisperModel

def normaliza(t):
    t = t.lower()
    t = re.sub(r"[^\wáéíóúüñ\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def audio_16k(ruta):
    # Se lee aquí y no con faster-whisper: su lector (PyAV) no casa con la versión de Colab
    a, sr = sf.read(str(ruta), dtype="float32")
    if a.ndim > 1:
        a = a.mean(axis=1)
    return librosa.resample(a, orig_sr=sr, target_sr=16000) if sr != 16000 else a

def cargar_cache(nombre):
    f = METRICAS / nombre
    return {(d["variante"], d["id"]): d for d in map(json.loads, f.read_text(encoding="utf-8").splitlines())} if f.exists() else {}

def anadir(nombre, fila):
    with (METRICAS / nombre).open("a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")

asr_hecho = cargar_cache("asr.jsonl")
asr = WhisperModel("large-v3", device="cuda", compute_type="float16")
for v, i in AUDIOS:
    if (v, i) in asr_hecho:
        continue
    segs, _ = asr.transcribe(audio_16k(comun.SALIDA / v / f"{i}.wav"), language="es", beam_size=5)
    hyp = " ".join(s.text.strip() for s in segs).strip()
    ref = " ".join(POR_ID[i]["frases"])
    fila = {"variante": v, "id": i, "wer": jiwer.wer(normaliza(ref), normaliza(hyp)) if normaliza(hyp) else 1.0,
            "transcripcion": hyp}
    anadir("asr.jsonl", fila)
    asr_hecho[(v, i)] = fila
del asr
comun.liberar()
print("WER hecho:", len(asr_hecho))

In [ ]:
# 4. Similitud de hablante con ECAPA-TDNN
import torch, torchaudio
from speechbrain.inference.speaker import EncoderClassifier
spk = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="/content/ecapa",
                                     run_opts={"device": "cuda"})

def embed(ruta):
    sig, sr = torchaudio.load(str(ruta))
    sig = sig.mean(0, keepdim=True)
    if sr != 16000:
        sig = torchaudio.functional.resample(sig, sr, 16000)
    with torch.no_grad():
        e = spk.encode_batch(sig.to("cuda")).squeeze()
    return e / e.norm()

# La identidad se compara siempre con las referencias normales de la voz (los items k también con naval/)
voz_ref = {}
for voz in sorted({it["voz"] for it in ITEMS}):
    clips = sorted((comun.REFS / voz).glob(f"{voz}_ref*.wav"))
    m = torch.stack([embed(c) for c in clips]).mean(0)
    voz_ref[voz] = m / m.norm()
    print(voz, "referencias:", [c.name for c in clips])

sim_hecho = cargar_cache("similitud.jsonl")
for v, i in AUDIOS:
    if (v, i) in sim_hecho:
        continue
    s = float((embed(comun.SALIDA / v / f"{i}.wav") * voz_ref[POR_ID[i]["voz"]]).sum())
    fila = {"variante": v, "id": i, "similitud": s}
    anadir("similitud.jsonl", fila)
    sim_hecho[(v, i)] = fila
del spk
comun.liberar()
print("Similitud hecha:", len(sim_hecho))

In [ ]:
# 5. Calidad estimada con UTMOSv2
import utmosv2
mos_modelo = utmosv2.create_model(pretrained=True)
mos_hecho = cargar_cache("utmos.jsonl")
for v, i in AUDIOS:
    if (v, i) in mos_hecho:
        continue
    fila = {"variante": v, "id": i, "utmos": float(mos_modelo.predict(input_path=str(comun.SALIDA / v / f"{i}.wav")))}
    anadir("utmos.jsonl", fila)
    mos_hecho[(v, i)] = fila
del mos_modelo
comun.liberar()
print("UTMOS hecho:", len(mos_hecho))

In [ ]:
# 6. Tablas
import pandas as pd
filas = []
for v, i in AUDIOS:
    reg = {}
    for l in (comun.SALIDA / v / "registro.jsonl").read_text(encoding="utf-8").splitlines():
        d = json.loads(l)
        if d["id"] == i and d.get("ok"):
            reg = d
    filas.append({"variante": v, "id": i, "voz": POR_ID[i]["voz"], "ref": POR_ID[i]["ref"],
                  "wer": asr_hecho[(v, i)]["wer"], "similitud": sim_hecho[(v, i)]["similitud"],
                  "utmos": mos_hecho[(v, i)]["utmos"], "seg_audio": reg.get("segundos_audio"),
                  "seg_gen": reg.get("segundos_generacion"), "transcripcion": asr_hecho[(v, i)]["transcripcion"]})
df = pd.DataFrame(filas)
df["alucinacion_probable"] = df["wer"] > 0.15
df.to_csv(METRICAS / "detalle.csv", index=False)

resumen = df.groupby("variante").agg(
    wer_medio=("wer", "mean"), items_con_wer_alto=("alucinacion_probable", "sum"),
    similitud=("similitud", "mean"), utmos=("utmos", "mean"),
    rtf=("seg_gen", "sum")).reset_index()
resumen["rtf"] = resumen["rtf"] / df.groupby("variante")["seg_audio"].sum().values
resumen = resumen.sort_values("wer_medio")
resumen.to_csv(METRICAS / "resumen.csv", index=False)
por_voz = df.pivot_table(index="variante", columns="voz", values=["wer", "similitud"], aggfunc="mean")
por_voz.to_csv(METRICAS / "por_voz.csv")
pd.set_option("display.width", 200)
print("=== RESUMEN (WER: menos es mejor; similitud y UTMOS: más es mejor; rtf<1 es más rápido que tiempo real)")
print(resumen.round(3).to_string(index=False))
print("\n=== POR VOZ")
print(por_voz.round(3).to_string())
print("\n=== Peores WER (posibles alucinaciones)")
print(df.sort_values("wer", ascending=False).head(15)[["variante", "id", "wer", "transcripcion"]].to_string(index=False))

In [ ]:
# 7. Página de escucha a ciegas
# Elige qué se escucha: 14 items representativos y las variantes que quieras (por defecto, todas).
ITEMS_ESCUCHA = ["n01", "n05", "n10", "n14", "n17", "n23", "n25", "v02", "v05", "v09", "b03", "b10", "k04", "k09"]
VARIANTES_ESCUCHA = VARIANTES

import base64, random, html, subprocess, tempfile
def mp3_b64(ruta):
    with tempfile.NamedTemporaryFile(suffix=".mp3") as t:
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(ruta), "-ac", "1", "-b:a", "64k", t.name], check=True)
        return base64.b64encode(open(t.name, "rb").read()).decode()

def audio_tag(ruta):
    return f'<audio controls preload="none" src="data:audio/mpeg;base64,{mp3_b64(ruta)}"></audio>'

CRITERIOS = [("parecido", "Parecido"), ("acento", "Acento nativo"), ("naturalidad", "Naturalidad"),
             ("alucinaciones", "Sin alucinaciones")]
rnd = random.Random(2026)
clave, bloques = {}, []
for i in ITEMS_ESCUCHA:
    it = POR_ID[i]
    vs = [v for v in VARIANTES_ESCUCHA if (comun.SALIDA / v / f"{i}.wav").exists()]
    rnd.shuffle(vs)
    letras = {chr(65 + n): v for n, v in enumerate(vs)}
    clave[i] = letras
    ref = sorted((comun.REFS / it["ref"]).glob(f"{it['ref']}_ref02.wav"))
    texto = html.escape(" ".join(it["frases"]))
    filas = []
    for letra, v in letras.items():
        selects = "".join(
            f'<label>{nom} <select data-item="{i}" data-letra="{letra}" data-criterio="{c}">'
            + "".join(f"<option>{x}</option>" for x in ["", "1", "2", "3", "4", "5"]) + "</select></label>"
            for c, nom in CRITERIOS)
        extra = ""
        if it["ref"] == "naval_kapil":
            gemelo = comun.SALIDA / v / f"{it['mismo_texto_que']}.wav"
            if gemelo.exists():
                extra = (f'<div class="gemelo">Misma voz con la referencia normal: {audio_tag(gemelo)}'
                         f'<label>Tono: ¿se nota el cambio hacia la referencia de Kapil? <select data-item="{i}" '
                         f'data-letra="{letra}" data-criterio="tono">'
                         + "".join(f"<option>{x}</option>" for x in ["", "1", "2", "3", "4", "5"]) + "</select></label></div>")
        filas.append(f'<div class="version"><b>{letra}</b> {audio_tag(comun.SALIDA / v / f"{i}.wav")}'
                     f'<div class="notas">{selects}</div>{extra}</div>')
    ancla = f'<div class="ancla">Referencia original ({it["ref"]}): {audio_tag(ref[0])}</div>' if ref else ""
    bloques.append(f'<section><h2>{i} · {it["voz"]}</h2><p class="texto">{texto}</p>{ancla}{"".join(filas)}</section>')

pagina = f'''<!doctype html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Escucha a ciegas</title>
<style>
body{{font-family:system-ui,sans-serif;max-width:900px;margin:0 auto;padding:16px;background:#fafafa;color:#222}}
section{{background:#fff;border:1px solid #ddd;border-radius:8px;padding:12px 16px;margin:16px 0}}
.texto{{font-style:italic}} .ancla{{color:#555;margin-bottom:8px}}
.version{{border-top:1px solid #eee;padding:8px 0}} .notas label{{margin-right:12px;white-space:nowrap}}
.gemelo{{margin-top:6px;padding-left:12px;border-left:3px solid #ccc}} audio{{vertical-align:middle;max-width:100%}}
#barra{{position:sticky;top:0;background:#222;color:#fff;padding:8px 12px;border-radius:6px}}
button{{font-size:15px;padding:4px 12px}}
</style></head><body>
<div id="barra">Puntúa de 1 (mal) a 5 (perfecto). Tus notas se guardan en este navegador.
<button onclick="descargar()">Descargar puntuaciones (CSV)</button> <span id="cuenta"></span></div>
<p>Cada versión es un motor distinto, con letras al azar en cada frase. Escucha primero la referencia original.
<b>Sin alucinaciones</b>: 5 = dice exactamente el texto; 1 = repite, se salta o inventa palabras.</p>
{"".join(bloques)}
<script>
const K = "criba_escucha_v1";
let notas = {{}};
try {{ notas = JSON.parse(localStorage.getItem(K) || "{{}}"); }} catch (e) {{}}
const sels = document.querySelectorAll("select");
function id(s) {{ return s.dataset.item + "|" + s.dataset.letra + "|" + s.dataset.criterio; }}
function contar() {{ document.getElementById("cuenta").textContent =
  Object.values(notas).filter(Boolean).length + " / " + sels.length + " notas"; }}
sels.forEach(s => {{ if (notas[id(s)]) s.value = notas[id(s)];
  s.addEventListener("change", () => {{ notas[id(s)] = s.value;
    try {{ localStorage.setItem(K, JSON.stringify(notas)); }} catch (e) {{}} contar(); }}); }});
contar();
function descargar() {{
  let csv = "item,letra,criterio,nota\\n";
  sels.forEach(s => {{ if (s.value) csv += [s.dataset.item, s.dataset.letra, s.dataset.criterio, s.value].join(",") + "\\n"; }});
  const a = document.createElement("a");
  a.href = URL.createObjectURL(new Blob([csv], {{type: "text/csv"}}));
  a.download = "puntuaciones_criba.csv"; a.click();
}}
</script></body></html>'''
ESCUCHA = comun.SALIDA / "escucha"
ESCUCHA.mkdir(exist_ok=True)
(ESCUCHA / "escucha.html").write_text(pagina, encoding="utf-8")
(ESCUCHA / "clave.json").write_text(json.dumps(clave, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Página: {ESCUCHA / 'escucha.html'} ({len(pagina) / 1e6:.1f} MB). La clave está aparte, en clave.json: no la abras antes de puntuar.")

In [ ]:
# 8. Desconectar la GPU
comun.desconectar()